# IFRS Agent — demo notebook

This notebook demonstrates the **IFRS/IAS standards agent**: a Gemini-powered
LangChain agent with two tools —

- `search_standards`: semantic search (FAISS) over the EU-endorsed texts of
  **IAS 2, IAS 16, IAS 36, IFRS 15, IFRS 16** (Regulation (EU) 2023/1803);
- `calculator`: exact arithmetic, so figures are computed, never guessed.

Unlike plain RAG (which always retrieves, then answers), the agent **decides**
at each step: search, calculate, chain both, or refuse a question outside its
five standards.

**Before running:** you need the one-time setup from the README (venv +
`pip install -r requirements.txt` + your `GOOGLE_API_KEY` in `.env`) and a
built index (`python bot_review.py --index_only`). Run this notebook from the
repository root. On the free Gemini tier, if a cell fails with a `429` quota
error, wait ~60 seconds and re-run that cell.

In [1]:
import os

print("Current directory:")
print(os.getcwd())

print("\nFiles:")
print(os.listdir())

Current directory:
C:\Users\lenovo\Documents\GitHub\ifrs-agent

Files:
['.env', '.env.example', '.git', '.gitignore', '.ipynb_checkpoints', 'agent.py', 'app.py', 'batch_query.py', 'bot_review.py', 'demo.ipynb', 'index', 'LICENSE', 'llm_as_judge.py', 'output_crawler', 'prepare_ifrs_data.py', 'rageval.py', 'README.md', 'requirements-pinned.txt', 'requirements.txt', 'TEAM_PLAN.md', 'tools.py', 'venv', '__pycache__']


In [2]:
# Setup: build the agent (verbose=True so we SEE its decisions)
from agent import build_agent

executor = build_agent(verbose=True)

def ask(question):
    """Ask the agent and print its final answer."""
    result = executor.invoke({"input": question})
    print("\n" + "=" * 70)
    print("FINAL ANSWER:\n")
    print(result["output"])

## Example 1 — definition question (pure retrieval)

Expected behaviour: the agent calls `search_standards`, finds IAS 2, and
answers with paragraph citations. No calculator needed.

In [3]:
ask("How are inventories measured under IAS 2, and what is net realisable value?")



> Entering new AgentExecutor chain...

Invoking: `search_standards` with `{'query': 'IAS 2 measurement of inventories and net realisable value definition'}`


[IAS 2 — Inventories — OBJECTIVE]
# IAS 2 — Inventories  
## OBJECTIVE  
**1** The objective of this standard is to prescribe the accounting treatment for inventories. A primary issue in accounting for inventories is the amount of cost to be recognised as an asset and carried forward until the related revenues are recognised. This standard provides guidance on the determination of cost and its subsequent recognition as an expense, including any write-down to net realisable value. It also provides guidance on the cost formulas that are used to assign costs to inventories.

---

[IAS 2 — Inventories — DEFINITIONS]
## DEFINITIONS  
**6** The following terms are used in this standard with the meanings specified: Inventories are assets: (a) held for sale in the ordinary course of business; (b) in the process of production for such s

## Example 2 — "which standard governs X?"

Expected behaviour: retrieval again — the agent should point to IFRS 16 and
describe the right-of-use asset + lease liability model.

In [4]:
ask("A company signs a 10-year office lease. Which standard applies and what must the lessee recognise?")



> Entering new AgentExecutor chain...

Invoking: `search_standards` with `{'query': 'IFRS 16 lessee recognition'}`


[IFRS 16 — Leases — OBJECTIVE]
# IFRS 16 — Leases  
## OBJECTIVE  
**1** This Standard sets out the principles for the recognition, measurement, presentation and disclosure of leases . The objective is to ensure that lessees and lessors provide relevant information in a manner that faithfully represents those transactions. This information gives a basis for users of financial statements to assess the effect that leases have on the financial position, financial performance and cash flows of an entity.  
**2** An entity shall consider the terms and conditions of contracts and all relevant facts and circumstances when applying this Standard. An entity shall apply this Standard consistently to contracts with similar characteristics and in similar circumstances.

---

[IFRS 16 — Leases — Recognition]
## Recognition  
**22** At the commencement date , a lessee shall recognis

## Example 3 — calculation question (both tools chained)

This is the agent's signature move. Expected behaviour: it first calls
`search_standards` to establish the treatment (IAS 16, straight-line
depreciation), **then** calls `calculator("(120000 - 20000) / 8")`, and
answers **12,500 euros** with the formula and citations.

In [5]:
ask("A machine costs 120000 euros, residual value 20000, useful life 8 years. "
    "What is the annual straight-line depreciation and which standard governs it?")



> Entering new AgentExecutor chain...

Invoking: `search_standards` with `{'query': 'straight-line depreciation'}`


[IAS 16 — Property, Plant and Equipment — Depreciation method]
## Depreciation method  
**60** The depreciation method used shall reflect the pattern in which the asset's future economic benefits are expected to be consumed by the entity.  
**61** The depreciation method applied to an asset shall be reviewed at least at each financial year - end and, if there has been a significant change in the expected pattern of consumption of the future economic benefits embodied in the asset, the method shall be changed to reflect the changed pattern. Such a change shall be accounted for as a change in an accounting estimate in accordance with IAS 8.  
**62** A variety of depreciation methods can be used to allocate the depreciable amount of an asset on a systematic basis over its useful life. These methods include the straight-line method, the diminishing balance method and the u

## Example 4 — impairment calculation (IAS 36)

Expected behaviour: search IAS 36 (recoverable amount = **higher** of fair
value less costs of disposal and value in use), then compute the loss:
recoverable amount 170,000 → impairment **30,000 euros**.

In [6]:
ask("An asset has a carrying amount of 200000 euros, fair value less costs of disposal "
    "of 150000 euros and value in use of 170000 euros. What is the impairment loss under IAS 36?")



> Entering new AgentExecutor chain...

Invoking: `calculator` with `{'expression': '200000 - 170000'}`
responded: The impairment loss is 30,000 euros.

Here's the calculation:
The recoverable amount of an asset is the higher of its fair value less costs of disposal and its value in use (IAS 36, paragraph 18).
In this case:
*   Fair value less costs of disposal: 150,000 euros
*   Value in use: 170,000 euros
The recoverable amount is the higher of these two, which is 170,000 euros.

An impairment loss is recognised when the carrying amount of an asset exceeds its recoverable amount (IAS 36, paragraph 59).


200000 - 170000 = 30000
Invoking: `search_standards` with `{'query': 'impairment loss calculation IAS 36'}`


[IAS 36 — Impairment of Assets — OBJECTIVE]
# IAS 36 — Impairment of Assets  
## OBJECTIVE  
**1** The objective of this standard is to prescribe the procedures that an entity applies to ensure that its assets are carried at no more than their recoverable amount. An asset is

## Example 5 — out-of-scope question (the agent must refuse)

Knowing your limits is part of being reliable. Expected behaviour: **no
invented answer** — the agent says this is outside its five standards.

In [7]:
ask("What does IFRS 9 say about expected credit losses?")



> Entering new AgentExecutor chain...
IFRS 9 is outside my coverage. I can only provide information on the following standards: IAS 2 Inventories, IAS 16 Property Plant and Equipment, IAS 36 Impairment of Assets, IFRS 15 Revenue from Contracts with Customers, and IFRS 16 Leases.

> Finished chain.

FINAL ANSWER:

IFRS 9 is outside my coverage. I can only provide information on the following standards: IAS 2 Inventories, IAS 16 Property Plant and Equipment, IAS 36 Impairment of Assets, IFRS 15 Revenue from Contracts with Customers, and IFRS 16 Leases.


## What this demonstrates

| Capability | Shown in |
|---|---|
| Retrieval-grounded answers with paragraph citations | Examples 1–2 |
| Autonomous tool choice and tool chaining (search → calculator) | Examples 3–4 |
| Exact arithmetic instead of LLM mental math | Examples 3–4 |
| Refusing out-of-scope questions instead of hallucinating | Example 5 |

The same behaviours are evaluated quantitatively over a 30-question test set
(`data/questions.xlsx`) with the evaluation scripts `rageval.py` and
`llm_as_judge.py` — see the README for results.